In [ ]:
from datetime import datetime
import base64
import io
import pandas as pd
import matplotlib.pyplot as plt
import pytz

In [ ]:
# Load the dataset
apps_df = pd.read_csv('Play Store Data.csv')
apps_df.head(3)

In [ ]:
# Helper function to parse size values to MB
def parse_size_to_mb(val):
  val = str(val).strip()

  if val.endswith('M') or val.endswith('m'):
    try:
      return float(val[:-1])
    except:
      return None

  elif val.endswith('k') or val.endswith('K'):
    try:
      return float(val[:-1]) / 1024.0
    except:
      return None

  return None

In [ ]:
# Remove duplicates
apps_df = apps_df.drop_duplicates()

# Drop missing values
apps_df = apps_df.dropna(subset=['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Last Updated']).copy()

# Clean Installs
apps_df['Installs'] = (
    apps_df['Installs']
    .astype(str)
    .str.replace('+', '', regex=False)
    .str.replace(',', '', regex=False)
)

apps_df = apps_df[apps_df['Installs'].str.isnumeric()].copy()
apps_df['Installs'] = apps_df['Installs'].astype(int)

# Clean Rating and Reviews
apps_df['Rating'] = pd.to_numeric(apps_df['Rating'], errors='coerce')
apps_df['Reviews'] = pd.to_numeric(apps_df['Reviews'], errors='coerce')

# Convert Size to MB
apps_df['Size_MB'] = apps_df['Size'].apply(parse_size_to_mb)

# Convert Last Updated to date
apps_df['Last Updated'] = pd.to_datetime(apps_df['Last Updated'], errors='coerce')

apps_df = apps_df.dropna(subset=['Rating', 'Reviews', 'Size_MB', 'Last Updated']).copy()

print(f'Rows after cleaning: {len(apps_df)}')
apps_df.head(3)

In [ ]:
# Apply filter conditions
cond_rating = apps_df['Rating'] >= 4.2
cond_no_numbers = ~apps_df['App'].astype(str).str.contains(r'\d', regex=True, na=False)
cond_category = apps_df['Category'].astype(str).str.startswith(('T', 'P'))
cond_reviews = apps_df['Reviews'] > 1000
cond_size = apps_df['Size_MB'].between(20, 80)

filtered_df = apps_df[
    cond_rating
    & cond_no_numbers
    & cond_category
    & cond_reviews
    & cond_size
].copy()

print(f'Total filtered rows: {len(filtered_df)}')
filtered_df[['App', 'Category', 'Rating', 'Reviews', 'Installs', 'Size_MB']].head(10)

In [ ]:
# Create monthly installs data
filtered_df['Month'] = filtered_df['Last Updated'].dt.to_period('M').dt.to_timestamp()

monthly_df = (
    filtered_df.groupby(['Month', 'Category'])['Installs']
    .sum()
    .reset_index()
    .sort_values(['Category', 'Month'])
)

# Calculate month-over-month growth
monthly_df['Growth'] = monthly_df.groupby('Category')['Installs'].pct_change() * 100
monthly_df['High_Growth'] = monthly_df['Growth'] > 25

monthly_df.head(10)

In [ ]:
# Translate required category names
category_translation_map = {
    'TRAVEL_AND_LOCAL': 'Voyages et infos locales',
    'PRODUCTIVITY': 'Productividad',
    'PHOTOGRAPHY': '写真'
}

def get_display_category(category):
  return category_translation_map.get(category, category.title())

monthly_df['Category_Display'] = monthly_df['Category'].apply(get_display_category)

In [ ]:
# Create cumulative installs data
pivot_df = monthly_df.pivot(
    index='Month',
    columns='Category',
    values='Installs'
).fillna(0).sort_index()

cumulative_df = pivot_df.cumsum()

# Sort categories by total installs
category_order = cumulative_df.iloc[-1].sort_values(ascending=False).index.tolist()
cumulative_df = cumulative_df[category_order]

cumulative_df.head()

In [ ]:
# Check time (4:00 PM to 6:00 PM IST)
ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist).time()

start_time = datetime.strptime('16:00:00', '%H:%M:%S').time()
end_time = datetime.strptime('18:00:00', '%H:%M:%S').time()

is_within_time = start_time <= current_time < end_time

graph_path = 'task_6_graph.html'
dashboard_path = 'task_6_dashboard.html'

In [ ]:
# Category summary for dashboard
category_summary = filtered_df.groupby('Category').agg(
    App_Count=('App', 'count'),
    Average_Rating=('Rating', 'mean'),
    Total_Installs=('Installs', 'sum')
).reset_index()

category_summary['Category_Display'] = category_summary['Category'].apply(get_display_category)

cat_rows_html = ''
for idx, row in category_summary.iterrows():
  cat_rows_html += f"<tr><td>{row['Category_Display']}</td><td>{int(row['App_Count'])}</td><td>{row['Average_Rating']:.2f}</td><td>{int(row['Total_Installs']):,}</td></tr>\n"

In [ ]:
# Generating the graph using Matplotlib and saving it as task_6_graph.html if within the allowed time window
if is_within_time:
  fig = plt.figure(figsize=(14, 7))
  ax = fig.add_subplot(111)

  dates = cumulative_df.index

  # Category labels
  labels = [
      get_display_category(category)
      for category in category_order
  ]

  # Category colors
  default_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
  category_colors = {
      category: default_colors[i % len(default_colors)]
      for i, category in enumerate(category_order)
  }

  # Create stacked area chart
  ax.stackplot(
      dates,
      [cumulative_df[category].values for category in category_order],
      labels=labels,
      colors=[category_colors[category] for category in category_order],
      alpha=0.45
  )

  # Calculate stack boundaries
  stack_top = cumulative_df.cumsum(axis=1)
  stack_bottom = stack_top - cumulative_df

  # Highlight months where growth is greater than 25%
  for category in category_order:
    category_growth_df = monthly_df[
        monthly_df['Category'] == category
    ].sort_values('Month').reset_index(drop=True)

    for i in range(1, len(category_growth_df)):
      if category_growth_df.loc[i, 'High_Growth']:
        month = category_growth_df.loc[i, 'Month']

        if month not in cumulative_df.index:
          continue

        month_index = cumulative_df.index.get_loc(month)

        if month_index == 0:
          continue

        previous_month = cumulative_df.index[month_index - 1]

        ax.fill_between(
            [previous_month, month],
            [stack_bottom.loc[previous_month, category], stack_bottom.loc[month, category]],
            [stack_top.loc[previous_month, category], stack_top.loc[month, category]],
            color=category_colors[category],
            alpha=0.95
        )

  # Graph formatting
  ax.set_title(
      'Cumulative Installs Over Time by App Category\n'
      'Darker Areas Indicate >25% Month-over-Month Growth'
  )

  ax.set_xlabel('Month')
  ax.set_ylabel('Cumulative Installs')

  tick_gap = max(1, len(dates) // 12)
  tick_dates = dates[::tick_gap]

  ax.set_xticks(tick_dates)
  ax.set_xticklabels(
      [date.strftime('%b %Y') for date in tick_dates],
      rotation=45,
      ha='right'
  )

  ax.ticklabel_format(style='plain', axis='y')
  ax.grid(axis='y', alpha=0.2)
  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)

  ax.legend(
      title='Category',
      bbox_to_anchor=(1.01, 1),
      loc='upper left',
      frameon=False
  )

  plt.tight_layout()

  # Save Matplotlib figure into task_6_graph.html as Base64 HTML
  img_buffer = io.BytesIO()
  plt.savefig(img_buffer, format='png', bbox_inches='tight')
  img_buffer.seek(0)

  img_base64 = base64.b64encode(img_buffer.getvalue()).decode('utf-8')
  plt.close()

  graph_html_content = f"""<!DOCTYPE html>
<html>
<head>
    <title>Task 6 Graph</title>
</head>
<body style="text-align: center; background-color: #f4f6f9; padding: 20px;">
    <h2>Cumulative Installs by App Category</h2>
    <p>Darker areas indicate months with more than 25% month-over-month growth.</p>
    <img src="data:image/png;base64,{img_base64}" alt="Matplotlib Graph" style="max-width: 100%; height: auto; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1);">
</body>
</html>"""

  with open(graph_path, 'w', encoding='utf-8') as f:
    f.write(graph_html_content)

  status_section_html = f"""
    <div class="card">
        <h2>Graph Status</h2>
        <p style="color: green; font-weight: bold;">Status: Graph Successfully Generated</p>
        <p>The graph was generated because the current time is between 4 PM and 6 PM IST.</p>
        <p><a href="{graph_path}" target="_blank" style="display:inline-block; padding:10px 15px; background-color:#1a73e8; color:white; text-decoration:none; border-radius:5px;">Open Graph</a></p>
    </div>
    """

else:
  status_section_html = """
  <div class="card">
      <h2>Graph Status</h2>
      <p style="color: red; font-weight: bold;">Status: Graph Restricted</p>
      <p>The graph is available only between 4 PM and 6 PM IST.</p>
  </div>
  """

In [ ]:
# Generate dashboard
dashboard_html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Play Store App Analysis Dashboard - Task 6</title>
    <style>
        body {{font-family: Arial, sans-serif; margin: 30px; background-color: #f4f6f9; color: #333;}}
        h1 {{color: #1a73e8;}}
        .card {{background: white; padding: 20px; margin-bottom: 20px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);}}
        table {{width: 100%; border-collapse: collapse; margin-top: 10px;}}
        th, td {{padding: 10px; border: 1px solid #ddd; text-align: left;}}
        th {{background-color: #f0f4f8;}}
    </style>
</head>
<body>
    <h1>Play Store App Analysis Dashboard - Task 6</h1>

    <div class="card">
        <h2>Dashboard Summary</h2>
        <p><b>Filters Applied:</b> Rating &ge; 4.2, Reviews &gt; 1,000, Size between 20 MB and 80 MB, Categories starting with T or P, and App Name should not contain numbers.</p>
        <p><b>Total Filtered Rows:</b> {len(filtered_df):,}</p>
    </div>

    <div class="card">
        <h2>Category Summary</h2>
        <table>
            <tr><th>Category</th><th>Apps</th><th>Average Rating</th><th>Total Installs</th></tr>
            {cat_rows_html}
        </table>
    </div>

    {status_section_html}

</body>
</html>
"""

with open(dashboard_path, 'w', encoding='utf-8') as f:
  f.write(dashboard_html_content)